## Bronze Layer: Raw Data Ingestion

Reads raw CSV files (uploaded to a Unity Catalog volume) into Spark DataFrames
and lands them as-is into Bronze Delta tables — no cleaning or transformation
at this stage, per the Bronze/Silver/Gold lakehouse pattern.

**Source:** Rossmann Store Sales dataset (train.csv, store.csv)
**Destination:** `/Volumes/retail_project/bronze/raw_files/`

In [0]:
# Path to your uploaded files
sales_path = "/Volumes/retail_project/bronze/raw_files/train.csv"
store_path = "/Volumes/retail_project/bronze/raw_files/store.csv"

# Read train.csv (historical sales)
df_sales_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(sales_path)
)

display(df_sales_raw)

### Read store metadata

In [0]:
df_store_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(store_path)
)

display(df_store_raw)

### Sanity checks
Confirm row counts and column names match the expected Rossmann schema
before writing to Bronze tables.

In [0]:
print("Sales rows:", df_sales_raw.count())
print("Sales columns:", df_sales_raw.columns)
print()
print("Store rows:", df_store_raw.count())
print("Store columns:", df_store_raw.columns)

### Write Bronze Delta tables
Land raw data into `retail_project.bronze.sales_raw` and
`retail_project.bronze.store_raw` — no transformations applied yet.

In [0]:
# Write sales data as Bronze Delta table
df_sales_raw.write.format("delta").mode("overwrite").saveAsTable(
    "retail_project.bronze.sales_raw"
)

# Write store metadata as Bronze Delta table
df_store_raw.write.format("delta").mode("overwrite").saveAsTable(
    "retail_project.bronze.store_raw"
)

print("Bronze tables written successfully.")

### Verify Bronze tables

In [0]:
display(spark.sql("SELECT * FROM retail_project.bronze.sales_raw LIMIT 10"))
display(spark.sql("SELECT * FROM retail_project.bronze.store_raw LIMIT 10"))

### Data quality check
Findings:
- `sales_raw`: 0 nulls across all 1,017,209 rows — fully clean.
- `store_raw`: 3 nulls in `CompetitionDistance`, 354 nulls each in
  `CompetitionOpenSinceMonth`/`CompetitionOpenSinceYear` — to be handled
  in the Silver layer.

In [0]:
# Check for nulls in key columns - useful to document for the Silver cleaning step
from pyspark.sql.functions import col, sum as spark_sum

null_counts_store = df_store_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_store_raw.columns
])
display(null_counts_store)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts_sales = df_sales_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_sales_raw.columns
])
display(null_counts_sales)

In [0]:
import requests
import time
import json

url = "https://archive-api.open-meteo.com/v1/archive"

years = [("2013-01-01", "2013-12-31"), ("2014-01-01", "2014-12-31"), ("2015-01-01", "2015-07-31")]

all_weather_data = []

for start, end in years:
    params = {
        "latitude": 52.52,
        "longitude": 13.41,
        "start_date": start,
        "end_date": end,
        "daily": "temperature_2m_mean,precipitation_sum",
        "timezone": "Europe/Berlin"
    }
    
    resp = requests.get(url, params=params)
    print(start, "to", end, "-", resp.status_code)
    
    if resp.status_code == 200:
        all_weather_data.append(resp.json())
    else:
        print("Error:", resp.text)
        break  # stop immediately if we hit a limit again, don't waste more calls
    
    time.sleep(10)  # spaced out to be safe

print(f"\nSuccessfully fetched {len(all_weather_data)} of {len(years)} year-chunks")

In [0]:
with open("/tmp/weather_data_cache.json", "w") as f:
    json.dump(all_weather_data, f)

print("Weather data cached to /tmp/weather_data_cache.json")

In [0]:
import pandas as pd

# Flatten all year-chunks into one list of daily records
weather_records = []

for chunk in all_weather_data:
    dates = chunk["daily"]["time"]
    temps = chunk["daily"]["temperature_2m_mean"]
    precip = chunk["daily"]["precipitation_sum"]
    
    for d, t, p in zip(dates, temps, precip):
        weather_records.append({"Date": d, "TemperatureMean": t, "PrecipitationSum": p})

# Convert to pandas first, then to Spark
weather_pdf = pd.DataFrame(weather_records)
print(weather_pdf.shape)
weather_pdf.head()

In [0]:
from pyspark.sql.functions import to_date, col

df_weather = spark.createDataFrame(weather_pdf)
df_weather = df_weather.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))

# Write as a permanent Bronze table — no more dependency on the API or /tmp cache
df_weather.write.format("delta").mode("overwrite").saveAsTable(
    "retail_project.bronze.weather_raw"
)

print("Weather Bronze table written.")
display(df_weather)